In [2]:
import sys
import os

project_root = "home/antoniocorvino/Projects/BuildingsExtraction/"

sys.path.append("..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

from src.models.factory import build_model

ModuleNotFoundError: No module named 'torch'

In [ ]:
runs_path = "/home/antoniocorvino/Projects/BuildingsExtraction/"

image_path = f"{runs_path}/experiments/example/images/310110.tif"

model_path = f"{runs_path}/runs/InriaBuildingDataset/"

model_paths = {
    "bce": f"{model_path}/unetLL_bce_dim256_n56000_bs32/best_model.pth",
    # "dice": f"{model_path}/unetLL_dice_dim256_n47088_bs32/best_model.pth",
    # "focal": f"{model_path}/unetLL_focal_dim256_n47088_bs32/best_model.pth",
}

arch = "unetLL"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
img = Image.open(image_path).convert("RGB")
img_np = np.array(img)

transform = transforms.ToTensor()
img_tensor = transform(img).unsqueeze(0).to(device)

In [ ]:
def infer_model(model_path):
    model = build_model(arch).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    with torch.no_grad():
        logits = model(img_tensor)
        probs = torch.sigmoid(logits)

    return probs.squeeze().cpu().numpy()

In [ ]:
probs = {}

for name, path in model_paths.items():
    print(f"Running {name}...")
    p = infer_model(path)
    probs[name] = p
    np.save(f"{runs_path}/experiments/example/{name}.npy", p)

print("Saved probability maps.")

In [ ]:
p_bce = np.load(f"{runs_path}/experiments/example/bce.npy")
# p_tversky = np.load(f"{runs_path}/experiments/example/tversky.npy")
# p_focal = np.load(f"{runs_path}/experiments/example/focal_tversky.npy")

In [ ]:
stack = np.stack([p_bce, p_tversky, p_focal], axis=0)

mean_map = np.mean(stack, axis=0)
var_map = np.var(stack, axis=0)

print("Mean range:", mean_map.min(), mean_map.max())
print("Variance range:", var_map.min(), var_map.max())

In [ ]:
def show_overlay(title, base_img, heatmap, cmap="jet", alpha=0.5):
    plt.figure(figsize=(6,6))
    plt.imshow(base_img)
    plt.imshow(heatmap, cmap=cmap, alpha=alpha)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
def show_overlay_legend(title, base_img, heatmap, cmap="jet", alpha=0.5,
                 vmin=0.0, vmax=1.0, cbar_label="Probability"):

    fig, ax = plt.subplots(figsize=(6, 6))

    ax.imshow(base_img)

    im = ax.imshow(
        heatmap,
        cmap=cmap,
        alpha=alpha,
        vmin=vmin,
        vmax=vmax
    )

    ax.set_title(title)
    ax.axis("off")

    # Add colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label)

    plt.show()

In [ ]:
show_overlay_legend("BCE", img_np, p_bce)
# show_overlay_legend("Dice", img_np, p_tversky)
# show_overlay_legend("Focal", img_np, p_focal)

In [ ]:
show_overlay("Mean Probability", img_np, mean_map)
show_overlay("Variance (Disagreement)", img_np, var_map, cmap="hot")

In [ ]:
binary = mean_map > 0.5
show_overlay("Thresholded Mean (>0.5)", img_np, binary.astype(float))